In [13]:
import pandas as pd

In [14]:
app_df = pd.read_csv("https://raw.githubusercontent.com/arawsardni/5---Gaung-Taqwa-Indraswara---Sandra-Triana-Nursyafri/refs/heads/main/Dataset/application_record.csv")
cred_df = pd.read_csv("https://raw.githubusercontent.com/arawsardni/5---Gaung-Taqwa-Indraswara---Sandra-Triana-Nursyafri/refs/heads/main/Dataset/credit_record.csv")

In [20]:
is_bad = (
    cred_df['STATUS']
    .isin(['3','4','5'])
    .groupby(cred_df['ID'])
    .max()  # Jika ada minimal 1 True, return 1
    .astype(int)
)

In [21]:
# 3. Hitung fitur temporal (lebih cepat)
credit_agg = cred_df.groupby('ID').agg(
    MONTHS_BALANCE_MIN=('MONTHS_BALANCE', 'min'),
    MONTHS_BALANCE_MAX=('MONTHS_BALANCE', 'max'),
    COUNT_LATE_LAST_12M=pd.NamedAgg(
        column='STATUS', 
        aggfunc=lambda x: ((x.isin(['0','1','2','3','4','5'])) & 
                          (cred_df.loc[x.index, 'MONTHS_BALANCE'] >= -12)).sum()
    )
).reset_index()

# Gabungkan target dan fitur temporal
credit_agg['TARGET'] = is_bad.values

# 4. Optimasi merge
merged_data = pd.merge(
    app_df,
    credit_agg,
    on='ID',
    how='inner'
)

# 5. Hitung fitur turunan
merged_data['CREDIT_HISTORY_LENGTH'] = (
    merged_data['MONTHS_BALANCE_MIN'] - merged_data['MONTHS_BALANCE_MAX']
)

In [22]:
credit_agg

,ID,MONTHS_BALANCE_MIN,MONTHS_BALANCE_MAX,COUNT_LATE_LAST_12M,TARGET
0,5001711,-3,0,3,0
1,5001712,-18,0,4,0
2,5001713,-21,0,0,0
3,5001714,-14,0,0,0
4,5001715,-59,0,0,0
...,...,...,...,...,...
45980,5150482,-28,-11,0,0
45981,5150483,-17,0,0,0
45982,5150484,-12,0,12,0
45983,5150485,-1,0,2,0


In [23]:
# 3. Merge Data
merged_df = app_df.merge(credit_agg, on='ID', how='inner')

# 4. Validasi Hasil Merge
print(f"Jumlah Data Awal (Application): {len(app_df)}")
print(f"Jumlah Data Setelah Merge: {len(merged_df)}")
print("\nDistribusi Target:")
print(merged_df['TARGET'].value_counts(normalize=True))

Jumlah Data Awal (Application): 438557
Jumlah Data Setelah Merge: 36457

Distribusi Target:
TARGET
0    0.991716
1    0.008284
Name: proportion, dtype: float64


In [24]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36457 entries, 0 to 36456
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   36457 non-null  int64  
 1   CODE_GENDER          36457 non-null  object 
 2   FLAG_OWN_CAR         36457 non-null  object 
 3   FLAG_OWN_REALTY      36457 non-null  object 
 4   CNT_CHILDREN         36457 non-null  int64  
 5   AMT_INCOME_TOTAL     36457 non-null  float64
 6   NAME_INCOME_TYPE     36457 non-null  object 
 7   NAME_EDUCATION_TYPE  36457 non-null  object 
 8   NAME_FAMILY_STATUS   36457 non-null  object 
 9   NAME_HOUSING_TYPE    36457 non-null  object 
 10  DAYS_BIRTH           36457 non-null  int64  
 11  DAYS_EMPLOYED        36457 non-null  int64  
 12  FLAG_MOBIL           36457 non-null  int64  
 13  FLAG_WORK_PHONE      36457 non-null  int64  
 14  FLAG_PHONE           36457 non-null  int64  
 15  FLAG_EMAIL           36457 non-null 

In [25]:
merged_df

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,...,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,MONTHS_BALANCE_MIN,MONTHS_BALANCE_MAX,COUNT_LATE_LAST_12M,TARGET
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,...,1,1,0,0,NaN,2.0,-15,0,0,0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,...,1,1,0,0,NaN,2.0,-14,0,1,0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,...,1,0,0,0,Security staff,2.0,-29,0,3,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,...,1,0,1,1,Sales staff,1.0,-4,0,2,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,...,1,0,1,1,Sales staff,1.0,-26,-22,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36452,5149828,M,Y,Y,0,315000.0,Working,Secondary / secondary special,Married,House / apartment,...,1,0,0,0,Managers,2.0,-11,0,4,1
36453,5149834,F,N,Y,0,157500.0,Commercial associate,Higher education,Married,House / apartment,...,1,0,1,1,Medicine staff,2.0,-23,0,8,1
36454,5149838,F,N,Y,0,157500.0,Pensioner,Higher education,Married,House / apartment,...,1,0,1,1,Medicine staff,2.0,-32,0,0,1
36455,5150049,F,N,Y,0,283500.0,Working,Secondary / secondary special,Married,House / apartment,...,1,0,0,0,Sales staff,2.0,-9,0,10,0
